<a href="https://colab.research.google.com/github/dawoodahmadbakhsh21-web/-house-price-prediction/blob/main/The_streamlit_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Install Streamlit if not already installed
!pip install streamlit --quiet

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


# ── Page config ────────────────────────────────────────────
st.set_page_config(
    page_title="House Price Prediction",
    page_icon="🏠",
    layout="wide"
)

# ── Load & train (cached so it only runs once) ─────────────
@st.cache_resource
def load_and_train():
    housing = fetch_california_housing()
    df = pd.DataFrame(housing.data, columns=housing.feature_names)
    df['Price'] = housing.target * 100000

    # Clean outliers
    Q1, Q3 = df['Price'].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    df = df[(df['Price'] >= Q1 - 1.5*IQR) & (df['Price'] <= Q3 + 1.5*IQR)]

    X = df.drop(columns=['Price'])
    y = df['Price']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    lr = LinearRegression().fit(X_train, y_train)
    rf = RandomForestRegressor(n_estimators=100, max_depth=15,
                                   random_state=42, n_jobs=-1).fit(X_train, y_train)

    lr_r2   = r2_score(y_test, lr.predict(X_test))
    rf_r2   = r2_score(y_test, rf.predict(X_test))

    # Calculate RMSE manually by taking the square root of MSE
    lr_rmse = np.sqrt(mean_squared_error(y_test, lr.predict(X_test)))
    rf_rmse = np.sqrt(mean_squared_error(y_test, rf.predict(X_test)))

    return df, X, lr, rf, lr_r2, rf_r2, lr_rmse, rf_rmse

df, X, lr_model, rf_model, lr_r2, rf_r2, lr_rmse, rf_rmse = load_and_train()

# ── Sidebar ────────────────────────────────────────────────
st.sidebar.title("🏠 House Price Prediction")
st.sidebar.markdown("**Final Year ML Project**")
page = st.sidebar.radio("Navigate", [
    "🏠 Predict Price",
    "📊 Data Exploration",
    "🤖 Model Performance",
    "ℹ️ About"
])

# ══════════════════════════════════════════════════════════
# PAGE 1 — PREDICT
# ══════════════════════════════════════════════════════════
if page == "🏠 Predict Price":
    st.title("🏠 House Price Predictor")
    st.markdown("Adjust the property details below to get an instant price prediction.")

    col1, col2 = st.columns([1, 1])

    with col1:
        st.subheader("Property Details")
        income     = st.slider("Median Income ($10k)",    0.5, 15.0, 5.0, 0.1)
        house_age  = st.slider("House Age (years)",       1,    52,   20)
        ave_rooms  = st.slider("Average Rooms",           1.0, 15.0, 5.0, 0.5)
        ave_bedrms = st.slider("Average Bedrooms",        1.0,  5.0, 1.0, 0.5)
        population = st.slider("Block Population",        100, 10000, 1500, 100)
        ave_occup  = st.slider("Average Occupants",       1.0, 10.0, 3.0, 0.5)

        region = st.selectbox("California Region", [
            "San Francisco Bay Area", "Los Angeles",
            "San Diego", "Sacramento", "Fresno (Central Valley)"
        ])
        region_coords = {
            "San Francisco Bay Area": (37.77, -122.42),
            "Los Angeles":            (34.05, -118.24),
            "San Diego":              (32.72, -117.16),
            "Sacramento":             (38.58, -121.49),
            "Fresno (Central Valley)": (36.74, -119.77),
        }
        lat, lon = region_coords[region]
        model_choice = st.radio("Model", ["Random Forest", "Linear Regression"])

    with col2:
        st.subheader("Prediction Result")
        inp = np.array([[income, house_age, ave_rooms,
                          ave_bedrms, population, ave_occup, lat, lon]])
        model  = rf_model if model_choice == "Random Forest" else lr_model
        rmse   = rf_rmse  if model_choice == "Random Forest" else lr_rmse
        price  = model.predict(inp)[0]
        low, high = max(0, price - rmse), price + rmse

        st.metric("Predicted Price", f"${price:,.0f}")
        st.info(f"Estimated range: ${low:,.0f} — ${high:,.0f}")
        st.markdown(f"**Model used:** {model_choice}")
        st.markdown(f"**Region:** {region}")
        st.markdown(f"**Income level:** ${income * 10000:,.0f}/year")

        st.divider()
        r2 = rf_r2 if model_choice == "Random Forest" else lr_r2
        st.caption(f"Model accuracy — R² Score: {r2:.4f}")

# ══════════════════════════════════════════════════════════
# PAGE 2 — EDA
# ══════════════════════════════════════════════════════════
elif page == "📊 Data Exploration":
    st.title("📊 Data Exploration")

    st.subheader("Dataset Preview")
    st.dataframe(df.head(10), use_container_width=True)

    col1, col2 = st.columns(2)
    with col1:
        st.subheader("Price Distribution")
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.hist(df['Price'], bins=50, color='#7F77DD', edgecolor='white')
        ax.set_xlabel("House Price (USD)")
        ax.set_ylabel("Count")
        st.pyplot(fig)

    with col2:
        st.subheader("Correlation Heatmap")
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.heatmap(df.corr(), annot=True, fmt='.2f',
                    cmap='RdYlGn', center=0, ax=ax, linewidths=0.5)
        st.pyplot(fig)

    st.subheader("Geographic Price Map")
    fig, ax = plt.subplots(figsize=(10, 5))
    sc = ax.scatter(df['Longitude'], df['Latitude'],
                   c=df['Price'], cmap='RdYlGn', alpha=0.4, s=8)
    plt.colorbar(sc, ax=ax, label='Price (USD)')
    ax.set_title("California House Price by Location")
    st.pyplot(fig)

# ══════════════════════════════════════════════════════════
# PAGE 3 — MODEL PERFORMANCE
# ══════════════════════════════════════════════════════════
elif page == "🤖 Model Performance":
    st.title("🤖 Model Performance")

    col1, col2 = st.columns(2)
    with col1:
        st.metric("Linear Regression R²", f"{lr_r2:.4f}")
        st.metric("Linear Regression RMSE", f"${lr_rmse:,.0f}")
    with col2:
        st.metric("Random Forest R²", f"{rf_r2:.4f}",
                  delta=f"+{rf_r2 - lr_r2:.4f} better")
        st.metric("Random Forest RMSE", f"${rf_rmse:,.0f}",
                  delta=f"-${lr_rmse - rf_rmse:,.0f} lower")

    st.subheader("Feature Importance (Random Forest)")
    importance_df = pd.DataFrame({
        'Feature': X.columns,
        'Importance': rf_model.feature_importances_
    }).sort_values('Importance', ascending=True)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(importance_df['Feature'], importance_df['Importance'],
           color='#1D9E75')
    ax.set_xlabel("Importance Score")
    st.pyplot(fig)

# ══════════════════════════════════════════════════════════
# PAGE 4 — ABOUT
# ══════════════════════════════════════════════════════════
elif page == "ℹ️ About":
    st.title("ℹ️ About This Project")
    st.markdown("""
    ### House Price Prediction System
    **Final Year Machine Learning Project**

    This app predicts California house prices using two ML models
    trained on the California Housing Dataset (20,640 records).

    | Detail | Info |
    |--------|------|
    | Dataset | California Housing (sklearn) |
    | Models | Linear Regression, Random Forest |
    | Best Model | Random Forest (~85% R²) |
    | Built with | Python, Streamlit, scikit-learn |
    | Platform | Streamlit Cloud |
    """)

2026-06-01 05:06:37.986 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-01 05:06:37.988 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-01 05:06:48.758 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-01 05:06:48.920 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-06-01 05:06:48.920 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-01 05:06:48.921 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-01 05:06:48.923 Thread 'MainThread'